# Narrative Mirror — Pilot 2, clean version: all four texts in one run

A cleaned re-implementation of `02_qwen_summary_qa_pilot.ipynb`. The original redefined one `data` variable per text,
so its saved outputs show only the last text run. Here each text has its own entry in `TEXTS`, the steps are functions,
and one loop runs everything and saves a single results table.

**Same method and settings as the original pilot**, so results should match `results.md`:
- Hidden states: Qwen2.5-1.5B (base), each sentence encoded on its own, last layer, mean-pooled.
- Model trajectory: projection onto the first→last sentence direction, normalized to 0–1.
- Divergence: |trajectory − author salience| per sentence.
- Summary and Q&A: Qwen2.5-1.5B-Instruct, greedy decoding (1-sentence summary; 1-sentence answers).
- Coverage: all-MiniLM-L6-v2 similarity between each sentence and the summary (≥0.55 preserved, 0.35–0.55 partial, <0.35 dropped).

**How to run:** Google Colab. CPU works; a free GPU runtime is faster. Run all cells top to bottom.
Outputs: `pilot2_sentences.csv` (one row per sentence) and `pilot2_qa.csv` (one row per question).

**Known limitations, kept on purpose for reproducibility** (see the last section): sentences are encoded without
context; the projection largely reflects position; the coverage thresholds can mislabel a preserved sentence as dropped.

In [ ]:
!pip install -q transformers torch sentence-transformers pandas matplotlib

## 1. Texts, author salience and questions
Salience (0–1 per sentence) was assigned by the author. `target_idx` is the sentence each question depends on.

In [ ]:
TEXTS = {
  "flower_pots": {
    "domain": "memoir",
    "data": [
      {
        "idx": 0,
        "text": "Later that week, I drove my aunt to the supermarket.",
        "salience": 0.05
      },
      {
        "idx": 1,
        "text": "I couldn't find a parking spot, so I waited in the car near a private house's parking.",
        "salience": 0.1
      },
      {
        "idx": 2,
        "text": "I knew the owners-two young sisters who had just lost their mum.",
        "salience": 0.85
      },
      {
        "idx": 3,
        "text": "I knew they wouldn't mind.",
        "salience": 0.15
      },
      {
        "idx": 4,
        "text": "Still, I kept glancing around, checking if they were home, if they could see me.",
        "salience": 0.25
      },
      {
        "idx": 5,
        "text": "I felt like I was intruding.",
        "salience": 0.3
      },
      {
        "idx": 6,
        "text": "After a while, I saw my aunt in the mirror, walking back with shopping bags.",
        "salience": 0.1
      },
      {
        "idx": 7,
        "text": "I felt relieved.",
        "salience": 0.15
      },
      {
        "idx": 8,
        "text": "Just before she reached the car, my eyes caught the corner of their parking space.",
        "salience": 0.35
      },
      {
        "idx": 9,
        "text": "A few large flower pots stood there.",
        "salience": 0.4
      },
      {
        "idx": 10,
        "text": "Neglected. Dry soil. The flowers had died, it looked like ashes.",
        "salience": 0.85
      },
      {
        "idx": 11,
        "text": "For a moment, a thought passed-almost judgment.",
        "salience": 0.5
      },
      {
        "idx": 12,
        "text": "But it didn't stay.",
        "salience": 0.6
      },
      {
        "idx": 13,
        "text": "It was replaced, quietly, by something else.",
        "salience": 0.75
      },
      {
        "idx": 14,
        "text": "Compassion.",
        "salience": 0.95
      }
    ],
    "full_text": "Later that week, I drove my aunt to the supermarket.\nI couldn't find a parking spot, so I waited in the car near a private house's parking.\nI knew the owners—two young sisters who had just lost their mum.\nI knew they wouldn't mind.\nStill, I kept glancing around, checking if they were home, if they could see me.\nI felt like I was intruding.\nAfter a while, I saw my aunt in the mirror, walking back with shopping bags.\nI felt relieved.\nJust before she reached the car, my eyes caught the corner of their parking space.\nA few large flower pots stood there.\nNeglected. Dry soil. The flowers had died, it looked like ashes.\nFor a moment, a thought passed—almost judgment.\nBut it didn't stay.\nIt was replaced, quietly, by something else.\nCompassion.",
    "questions": [
      {
        "q": "Why were the flowers neglected?",
        "target_idx": 2
      },
      {
        "q": "What emotion did the narrator ultimately feel?",
        "target_idx": 14
      },
      {
        "q": "What did the dead flowers look like?",
        "target_idx": 10
      },
      {
        "q": "Was the narrator driving alone?",
        "target_idx": 0
      },
      {
        "q": "Did the narrator feel comfortable parking there?",
        "target_idx": 5
      }
    ]
  },
  "pomelo": {
    "domain": "memoir",
    "data": [
      {
        "idx": 0,
        "text": "As a child, I wanted money for one thing only: sweets.",
        "salience": 0.1
      },
      {
        "idx": 1,
        "text": "The latest ones at the village supermarket—those brightly wrapped treasures lined like jewels on the shelf.",
        "salience": 0.1
      },
      {
        "idx": 2,
        "text": "Eid was my payday.",
        "salience": 0.15
      },
      {
        "idx": 3,
        "text": "Dressed in fresh clothes and armed with a sense of entitlement only children possess, we roamed the village from house to house, palms open, smiles wide.",
        "salience": 0.15
      },
      {
        "idx": 4,
        "text": "No one owed us a thing, but it was a day designed to delight children, and we never questioned it.",
        "salience": 0.2
      },
      {
        "idx": 5,
        "text": "Relatives were the big banks—where bonuses dropped like gold coins.",
        "salience": 0.1
      },
      {
        "idx": 6,
        "text": "By midday, our pockets sagged with clinking change, and by evening we'd collapse on our beds, emptying wallets onto the blanket in gleeful chaos.",
        "salience": 0.1
      },
      {
        "idx": 7,
        "text": "We'd count, sort, and then guard that stash as if it were sacred.",
        "salience": 0.15
      },
      {
        "idx": 8,
        "text": "That treasure would fund our candy runs for months.",
        "salience": 0.1
      },
      {
        "idx": 9,
        "text": "There was a secret shortcut to the candy store that only the neighbourhood kids knew—a narrow path threading through someone's private garden.",
        "salience": 0.25
      },
      {
        "idx": 10,
        "text": "And in that garden stood a pomelo tree, its fruits impossibly large, suspended like golden moons.",
        "salience": 0.35
      },
      {
        "idx": 11,
        "text": "They called to us.",
        "salience": 0.3
      },
      {
        "idx": 12,
        "text": "We never dared pluck them, but the temptation lingered, sweet and sticky like the candies we craved.",
        "salience": 0.4
      },
      {
        "idx": 13,
        "text": "Months later, near the end of my stash, I passed by alone.",
        "salience": 0.45
      },
      {
        "idx": 14,
        "text": "The fruit still hung there, ripe with allure.",
        "salience": 0.35
      },
      {
        "idx": 15,
        "text": "I reached out, just to touch.",
        "salience": 0.55
      },
      {
        "idx": 16,
        "text": "My fingers closed around one.",
        "salience": 0.6
      },
      {
        "idx": 17,
        "text": "And then a voice—a woman's voice—startled me.",
        "salience": 0.65
      },
      {
        "idx": 18,
        "text": "She'd seen everything.",
        "salience": 0.7
      },
      {
        "idx": 19,
        "text": "She asked me to open my hand.",
        "salience": 0.8
      },
      {
        "idx": 20,
        "text": "I did.",
        "salience": 0.6
      },
      {
        "idx": 21,
        "text": "She didn't yell.",
        "salience": 0.45
      },
      {
        "idx": 22,
        "text": "She didn't scold.",
        "salience": 0.45
      },
      {
        "idx": 23,
        "text": "She simply took the last coins I had and walked away.",
        "salience": 0.85
      },
      {
        "idx": 24,
        "text": "That day, something shifted in me.",
        "salience": 0.75
      },
      {
        "idx": 25,
        "text": "It wasn't about the money, nor the fruit.",
        "salience": 0.65
      },
      {
        "idx": 26,
        "text": "It was the first time I've been treated like an adult, by an adult.",
        "salience": 0.8
      },
      {
        "idx": 27,
        "text": "Paying the price for a crime.",
        "salience": 0.7
      },
      {
        "idx": 28,
        "text": "I realised not everything comes free.",
        "salience": 0.75
      },
      {
        "idx": 29,
        "text": "I think that was my first real lesson in accountability.",
        "salience": 0.85
      },
      {
        "idx": 30,
        "text": "In the unseen moral lines that exist even when no one else is watching.",
        "salience": 0.9
      },
      {
        "idx": 31,
        "text": "That moment planted a seed in me.",
        "salience": 0.8
      },
      {
        "idx": 32,
        "text": "I'm being watched always.",
        "salience": 0.85
      },
      {
        "idx": 33,
        "text": "I lost coins but gained compass.",
        "salience": 0.9
      },
      {
        "idx": 34,
        "text": "Taqwa.",
        "salience": 0.95
      }
    ],
    "full_text": "As a child, I wanted money for one thing only: sweets. The latest ones at the village supermarket—those brightly wrapped treasures lined like jewels on the shelf. Eid was my payday. Dressed in fresh clothes and armed with a sense of entitlement only children possess, we roamed the village from house to house, palms open, smiles wide. No one owed us a thing, but it was a day designed to delight children, and we never questioned it.\nRelatives were the big banks—where bonuses dropped like gold coins. By midday, our pockets sagged with clinking change, and by evening we'd collapse on our beds, emptying wallets onto the blanket in gleeful chaos. We'd count, sort, and then guard that stash as if it were sacred. That treasure would fund our candy runs for months.\nThere was a secret shortcut to the candy store that only the neighbourhood kids knew—a narrow path threading through someone's private garden. And in that garden stood a pomelo tree, its fruits impossibly large, suspended like golden moons. They called to us. We never dared pluck them, but the temptation lingered, sweet and sticky like the candies we craved.\nMonths later, near the end of my stash, I passed by alone. The fruit still hung there, ripe with allure. I reached out, just to touch. My fingers closed around one. And then a voice—a woman's voice—startled me. She'd seen everything. She asked me to open my hand. I did. She didn't yell. She didn't scold. She simply took the last coins I had and walked away.\nThat day, something shifted in me. It wasn't about the money, nor the fruit. It was the first time I've been treated like an adult, by an adult. Paying the price for a crime. I realised not everything comes free. I think that was my first real lesson in accountability. In the unseen moral lines that exist even when no one else is watching. That moment planted a seed in me. I'm being watched always. I lost coins but gained compass. Taqwa.",
    "questions": [
      {
        "q": "Why did the woman take the child's coins?",
        "target_idx": 23
      },
      {
        "q": "What lesson did the narrator learn from this experience?",
        "target_idx": 29
      },
      {
        "q": "What does Taqwa mean in this context?",
        "target_idx": 34
      },
      {
        "q": "What did the children collect money for during Eid?",
        "target_idx": 0
      },
      {
        "q": "Did the children ever pick the pomelos before this incident?",
        "target_idx": 12
      },
      {
        "q": "Why didn't the woman yell or scold the child? What was the significance of her quiet response?",
        "target_idx": 26
      }
    ]
  },
  "legal": {
    "domain": "legal",
    "data": [
      {
        "idx": 0,
        "text": "The borrower agrees to repay the principal amount of $500,000 over a period of 10 years.",
        "salience": 0.3
      },
      {
        "idx": 1,
        "text": "Monthly payments shall be made on the first business day of each month.",
        "salience": 0.2
      },
      {
        "idx": 2,
        "text": "The interest rate is fixed at 4.5% per annum for the duration of the loan.",
        "salience": 0.3
      },
      {
        "idx": 3,
        "text": "Early repayment of the loan carries a penalty of 3% of the outstanding balance.",
        "salience": 0.85
      },
      {
        "idx": 4,
        "text": "The borrower must maintain a minimum account balance of $50,000 at all times.",
        "salience": 0.75
      },
      {
        "idx": 5,
        "text": "Failure to maintain the minimum balance constitutes a breach of covenant.",
        "salience": 0.9
      }
    ],
    "full_text": "The borrower agrees to repay the principal amount of $500,000 over a period of 10 years. Monthly payments shall be made on the first business day of each month. The interest rate is fixed at 4.5% per annum for the duration of the loan. Early repayment of the loan carries a penalty of 3% of the outstanding balance. The borrower must maintain a minimum account balance of $50,000 at all times. Failure to maintain the minimum balance constitutes a breach of covenant.",
    "questions": [
      {
        "q": "Can the borrower repay the loan early without any consequences?",
        "target_idx": 3
      },
      {
        "q": "What happens if the borrower's account drops below $50,000?",
        "target_idx": 5
      },
      {
        "q": "What is the interest rate?",
        "target_idx": 2
      }
    ]
  },
  "medical": {
    "domain": "medical",
    "data": [
      {
        "idx": 0,
        "text": "Patient is a 45-year-old female presenting with recurring headaches over the past two weeks.",
        "salience": 0.3
      },
      {
        "idx": 1,
        "text": "She reports the headaches are worse in the morning and improve throughout the day.",
        "salience": 0.4
      },
      {
        "idx": 2,
        "text": "She has a documented allergy to penicillin and sulfa drugs.",
        "salience": 0.85
      },
      {
        "idx": 3,
        "text": "Current medications include lisinopril 10mg daily for hypertension.",
        "salience": 0.6
      },
      {
        "idx": 4,
        "text": "She mentioned feeling increasingly hopeless and having difficulty sleeping.",
        "salience": 0.95
      },
      {
        "idx": 5,
        "text": "Physical examination reveals blood pressure of 138/88 and normal neurological findings.",
        "salience": 0.3
      }
    ],
    "full_text": "Patient is a 45-year-old female presenting with recurring headaches over the past two weeks. She reports the headaches are worse in the morning and improve throughout the day. She has a documented allergy to penicillin and sulfa drugs. Current medications include lisinopril 10mg daily for hypertension. She mentioned feeling increasingly hopeless and having difficulty sleeping. Physical examination reveals blood pressure of 138/88 and normal neurological findings.",
    "questions": [
      {
        "q": "Is it safe to prescribe amoxicillin to this patient?",
        "target_idx": 2
      },
      {
        "q": "Are there any mental health concerns mentioned in the record?",
        "target_idx": 4
      },
      {
        "q": "What is the patient's blood pressure?",
        "target_idx": 5
      }
    ]
  }
}

## 2. Load models

In [ ]:
import torch, numpy as np, pandas as pd
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "Qwen/Qwen2.5-1.5B"            # hidden states
CHAT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # summaries and answers
EMBEDDER   = "sentence-transformers/all-MiniLM-L6-v2"

base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModel.from_pretrained(BASE_MODEL, output_hidden_states=True, torch_dtype=torch.float32).to(DEVICE).eval()

chat_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, torch_dtype=torch.float32).to(DEVICE).eval()

embedder = SentenceTransformer(EMBEDDER, device=DEVICE)
print("Models loaded on", DEVICE)

## 3. Pipeline steps

In [ ]:
def sentence_vectors(sentences):
    """Each sentence encoded on its own (no context); last layer, mean over tokens."""
    vecs = []
    for s in sentences:
        inputs = base_tok(s, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = base(**inputs)
        vecs.append(out.hidden_states[-1].mean(dim=1).squeeze().float().cpu().numpy())
    return np.stack(vecs)

def trajectory(vectors):
    """Projection onto the first→last sentence direction, normalized to 0–1."""
    direction = vectors[-1] - vectors[0]
    direction = direction / np.linalg.norm(direction)
    proj = (vectors - vectors[0]) @ direction
    return (proj - proj.min()) / (proj.max() - proj.min())

def generate(prompt, max_new_tokens):
    text = chat_tok.apply_chat_template([{"role": "user", "content": prompt}],
                                        tokenize=False, add_generation_prompt=True)
    inputs = chat_tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        ids = chat.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return chat_tok.decode(ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def summarize(full_text):
    return generate(f"Summarize the following text in 1 sentence:\n\n{full_text}\n", max_new_tokens=80)

def answer(full_text, question):
    return generate(f"Based on this text:\n{full_text}\n\nAnswer in one sentence: {question}", max_new_tokens=60)

def coverage(sentences, summary):
    """Similarity of each sentence to the summary -> preserved / partial / dropped."""
    summ = embedder.encode(summary, convert_to_tensor=True)
    rows = []
    for s in sentences:
        sim = util.cos_sim(embedder.encode(s, convert_to_tensor=True), summ).item()
        status, drop = ("preserved", 0.0) if sim >= 0.55 else ("partial", 0.5) if sim >= 0.35 else ("dropped", 1.0)
        rows.append((round(sim, 3), status, drop))
    return rows

## 4. Run all four texts

In [ ]:
sentence_rows, qa_rows, summaries = [], [], {}

for name, t in TEXTS.items():
    sentences = [d["text"] for d in t["data"]]
    salience  = np.array([d["salience"] for d in t["data"]])

    traj = trajectory(sentence_vectors(sentences))
    div  = np.abs(traj - salience)

    summary = summarize(t["full_text"])
    summaries[name] = summary
    cov = coverage(sentences, summary)

    for i, s in enumerate(sentences):
        sim, status, drop = cov[i]
        sentence_rows.append(dict(text_id=name, domain=t["domain"], idx=i, sentence=s,
                                  salience=salience[i], trajectory=round(float(traj[i]), 3),
                                  divergence=round(float(div[i]), 3),
                                  coverage_similarity=sim, coverage_status=status, drop_score=drop))

    for qa in t["questions"]:
        qa_rows.append(dict(text_id=name, question=qa["q"], target_idx=qa["target_idx"],
                            target_divergence=round(float(div[qa["target_idx"]]), 3),
                            answer=answer(t["full_text"], qa["q"]),
                            verdict=""))   # fill in by hand: correct / partial / wrong

    print(f"\n=== {name} ===\nSUMMARY: {summary}")

sent_df = pd.DataFrame(sentence_rows)
qa_df   = pd.DataFrame(qa_rows)

## 5. Results per text

In [ ]:
for name in TEXTS:
    d = sent_df[sent_df.text_id == name]
    corr = d["divergence"].corr(d["drop_score"])
    print(f"\n=== {name} ({len(d)} sentences) ===")
    print("Highest divergence:")
    print(d.nlargest(3, "divergence")[["idx", "divergence", "salience", "sentence"]].to_string(index=False))
    print(f"Coverage correlation (divergence vs drop score): {corr:.3f}")
    print("Q&A:")
    print(qa_df[qa_df.text_id == name][["target_idx", "target_divergence", "question", "answer"]].to_string(index=False))

sent_df.to_csv("pilot2_sentences.csv", index=False)
qa_df.to_csv("pilot2_qa.csv", index=False)
print("\nSaved pilot2_sentences.csv and pilot2_qa.csv — add your correct / partial / wrong verdicts to pilot2_qa.csv.")

## 6. Plots: model trajectory vs author salience

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(TEXTS), 1, figsize=(10, 3.2 * len(TEXTS)))
for ax, name in zip(axes, TEXTS):
    d = sent_df[sent_df.text_id == name]
    ax.plot(d["idx"], d["trajectory"], "o-", label="model trajectory")
    ax.plot(d["idx"], d["salience"], "s--", label="author salience")
    ax.set_title(name); ax.set_ylim(-0.05, 1.05); ax.set_xlabel("sentence"); ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig("pilot2_trajectories.png", dpi=120); plt.show()

## 7. Limitations and next steps

**Kept on purpose so this reproduces the pilot:**
- Sentences are encoded one at a time, without the text before them, so this is not a contextual measure.
- The first→last projection pins the first sentence near 0 and the last near 1, so it largely reflects position.
- |trajectory − salience| treats under-weighting and over-weighting the same.
- The coverage thresholds can mislabel a preserved sentence as dropped (the medical "hopelessness" sentence in the original run). Check labels by hand.
- One run per text, greedy decoding, very small n; author-only salience.

**Next:** measure each sentence's effect on the model's state *in context* (state before vs after the sentence), keep the
sign of the gap, use texts with implied meaning plus explicit controls, hand-checked failure labels, reader annotations,
and larger open models.